# Protein–ligand interactions across a trajectory (ProLIF)

Downloads a topology + trajectory from Zenodo, runs a ProLIF interaction
fingerprint over the trajectory via `openfe_analysis.prolif.ProLIFAnalysis`, and
shows the plots (2D ligand network per frame and aggregated, barcode across frames,
3D py3Dmol).

For a single static structure, including water-mediated interactions, see
`prolif_from_structure.ipynb`.

**Environment** (one-time, from the repo root). Create the base env from **conda-forge**
(mixing channels breaks RDKit's DLLs on Windows) and launch Jupyter from the
**activated** env:

```bash
conda create    -n openfe_analysis_env -c conda-forge --override-channels python=3.13 -y
conda env update -n openfe_analysis_env -f environment.yml
conda install   -n openfe_analysis_env -c conda-forge --override-channels jupyterlab py3dmol -y
conda activate openfe_analysis_env
pip install -e . --no-deps
```

> **`py3Dmol`** is required for the 3D view and is **not** pulled in by ProLIF — install it explicitly (above).

In [ ]:
import pathlib

import pooch
import MDAnalysis as mda

from openfe_analysis.reader import FEReader
from openfe_analysis.prolif import ProLIFAnalysis
from openfe_analysis.utils.plotting import (
    plot_prolif_lignetwork,
    plot_prolif_barcode,
    plot_prolif_3d,
)

## 1. Download the data

Fetches `openfe_analysis_full.tar.gz` (topology `hybrid_system.pdb` + trajectory
`simulation.nc`) from Zenodo record
[10.5281/zenodo.22828040](https://doi.org/10.5281/zenodo.22828040). Cached after
the first run, so re-running is instant.

In [ ]:
ZENODO_DOI = "doi:10.5281/zenodo.22828040"
ARCHIVE = "openfe_analysis_full.tar.gz"
registry = {ARCHIVE: "md5:a51b1f8d98b91ab1a69a6f55508d07db"}

fetcher = pooch.create(
    path=pooch.os_cache("openfe_analysis"),
    base_url=ZENODO_DOI,
    registry=registry,
)
fetcher.fetch(ARCHIVE, processor=pooch.Untar())

data_dir = (
    pathlib.Path(pooch.os_cache("openfe_analysis"))
    / (ARCHIVE + ".untar")
    / "openfe_analysis_full"
)
pdb = data_dir / "hybrid_system.pdb"
nc = data_dir / "simulation.nc"
print("topology:", pdb, pdb.exists())
print("trajectory:", nc, nc.exists())

## 2. Build the Universe and select the ligand

In [ ]:
u = mda.Universe(pdb, nc, format=FEReader, index=0)
print("n_frames:", len(u.trajectory))
print("resnames:", sorted(set(u.residues.resnames)))

In [ ]:
# The alchemical ligand in this dataset is resname 'UNK' (36 atoms).
# If the printout above shows a different ligand resname, adjust the selection.
ligand_ag = u.select_atoms("resname UNK")
print("ligand atoms:", ligand_ag.n_atoms)

## 3. Run the fingerprint

`interactions=None` uses ProLIF's default set (Hydrophobic, HBDonor, HBAcceptor,
PiStacking, Anionic, Cationic, CationPi, PiCation, VdWContact). Only the first 10
frames are analysed here to keep it quick.

In [ ]:
analysis = ProLIFAnalysis(
    u,
    ligand_ag,
    interactions=None,
    guess_bonds=True,
)
analysis.run(stop=10, step=1, n_jobs=1, progress=True)

In [ ]:
df = analysis.to_dataframe()
print("frames x interactions:", df.shape)
df.head()

## 4. 2D ligand-interaction network

### Single frame

By default the first analysed frame is shown (`kind="frame"`).

In [ ]:
plot_prolif_lignetwork(analysis.fp, analysis.ligand_ag)

### Aggregated over frames

`kind="aggregate"` shows every interaction present in at least `threshold` of the
analysed frames (here 30%) — usually the more useful summary for a trajectory.

In [ ]:
plot_prolif_lignetwork(
    analysis.fp, analysis.ligand_ag, kind="aggregate", threshold=0.3
)

## 5. Interaction barcode across frames

In [ ]:
plot_prolif_barcode(analysis.fp)

## 6. 3D interaction view (py3Dmol)

Renders the ligand, pocket residues and any nearby waters with the detected
interactions for the chosen `frame`. Requires `py3Dmol` (see setup note).

In [ ]:
plot_prolif_3d(
    analysis.fp,
    analysis.ligand_ag,
    analysis.protein_ag,
    analysis.water_ag,
    frame=0,
)

## 7. Choosing which interactions to compute

`interactions` controls what is computed:

- `None` (default): ProLIF's default set (used above).
- a list, e.g. `["HBDonor", "HBAcceptor"]`: only those — faster and easier to read.
- `"all"`: every available interaction, including bridged ones such as `WaterBridge`.

Bonds were already guessed on this Universe above, so `guess_bonds=False` skips
redoing that work.

In [ ]:
hbonds = ProLIFAnalysis(
    u, ligand_ag, interactions=["HBDonor", "HBAcceptor"], guess_bonds=False
)
hbonds.run(stop=10, n_jobs=1, progress=False)
hbonds.to_dataframe().head()

`"all"` requests every available interaction. Two caveats for MD systems:

- `WaterBridge` needs water — this system has none, so it is dropped with a warning
  (see `prolif_from_structure.ipynb` for a solvated example).
- `ImplicitHBAcceptor` / `ImplicitHBDonor` are meant for structures **without
  explicit hydrogens**; on explicit-H systems like this one they fail at `run()`
  (RDKit hybridization is not assigned).

So here we only list what `"all"` would track — for actual runs, prefer an
explicit list.

In [ ]:
all_ifp = ProLIFAnalysis(u, ligand_ag, interactions="all", guess_bonds=False)
sorted(all_ifp.fp.interactions)